# Optimize And Validate Readout Amplitudes

This notebook can run a fresh readout amplitude optimization and then validate the optimizer amplitudes against the `main` profile.

Flow:

1. Optionally run `ReadoutAmplitudeSweepWorkflow` to produce an optimizer run folder.
2. Load the optimizer run summary from `summary.json`.
3. Validate every optimizer amplitude with `ReadoutOptimizerValidation`.
4. Save validation CSV, JSON, Markdown report, and figure under `OPTIMIZER_RUN_DIR/validation/main_profile_iq_sweep/`.

The readout workflow runs kernels for the full qubit list in one multi-qubit kernel experiment instance.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

WORKBENCH_ROOT = Path.cwd()
if str(WORKBENCH_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKBENCH_ROOT))

from optimize.readout.optimizer import ReadoutAmplitudeSweepSettings, ReadoutAmplitudeSweepWorkflow
from optimize.readout.optimizer.scan_types import ReadoutScanMethod
from optimize.readout.readout_workflow import ReadoutFidelityWorkflowSettings
from optimize.readout.validation import ReadoutOptimizerValidation, ReadoutOptimizerValidationSettings
from qratena.system.components_params.reset_settings import ResetSettings
from qratena.util.enums import ResetType
from resources.load_profile import load_profile, load_task_manager


## Configuration

Use `RUN_OPTIMIZER = True` for a fresh run. Use `False` with `EXISTING_OPTIMIZER_RUN_DIR` to validate an already saved run.


In [ ]:
RUN_OPTIMIZER = True
EXISTING_OPTIMIZER_RUN_DIR = Path("data/readout_optimize/YYYY-MM-DD/HH-MM-SS_sweep_q1_q2")

PROFILE_NAME = "main"
QUBIT_NAMES = ["q1", "q2"]
OPTIMIZATION_AMPLITUDES = np.linspace(0.005, 0.11, 30)
OPTIMIZATION_OUTPUT_DIR = Path("data/readout_optimize")
OPTIMIZATION_METHOD = ReadoutScanMethod.SWEEP
LOW_PRIORITY_TASKS = True

ACTIVE_RESET_NUM = 5
TASK_STATUS_POLL_INTERVAL = 10.0
SHOW_HANDLER_OUTPUT = False
DO_EMULATION = False
STATES = ["g", "e"]


## Run Optimization

This cell creates the optimizer run folder used by validation. Skip it by setting `RUN_OPTIMIZER = False`.


In [ ]:
if RUN_OPTIMIZER:
    optimization_profile = load_profile(PROFILE_NAME)
    if STATES == ["g", "e", "f"]:
        optimization_profile.ensure_pi_ef_pulse_for_all_qubits(overwrite=False)

    optimization_task_manager = object() if DO_EMULATION else load_task_manager()
    optimization_workflow_settings = ReadoutFidelityWorkflowSettings(
        profile_name=PROFILE_NAME,
        do_emulation=DO_EMULATION,
        run_resonator=False,
        run_kernels=True,
        run_iq_blobs=True,
        do_plotting=False,
        show_handler_output=SHOW_HANDLER_OUTPUT,
        report_timing=True,
        task_status_poll_interval=TASK_STATUS_POLL_INTERVAL,
        low_priority_tasks=LOW_PRIORITY_TASKS,
        reset=ResetSettings(reset_type=ResetType.ACTIVE, reset_num=ACTIVE_RESET_NUM),
        states=STATES,
    )
    optimizer_settings = ReadoutAmplitudeSweepSettings(
        amplitudes=OPTIMIZATION_AMPLITUDES,
        method=OPTIMIZATION_METHOD,
        auto_save_results=True,
        use_live_html_plotter=True,
        live_html_output_dir=OPTIMIZATION_OUTPUT_DIR,
        workflow_settings=optimization_workflow_settings,
    )
    optimizer = ReadoutAmplitudeSweepWorkflow(
        qubit_names=QUBIT_NAMES,
        profile=optimization_profile,
        task_manager=optimization_task_manager,
        settings=optimizer_settings,
    )
    optimizer.run()
    OPTIMIZER_RUN_DIR = Path(optimizer.run_dir)
else:
    OPTIMIZER_RUN_DIR = EXISTING_OPTIMIZER_RUN_DIR

print(f"Optimizer run dir: {OPTIMIZER_RUN_DIR}")


## Validate Optimizer Amplitudes

This applies every optimizer amplitude to a fresh `main` profile copy and runs one readout workflow per amplitude for the optimizer qubits.


In [ ]:
validation_profile = load_profile(PROFILE_NAME)
validation_task_manager = object() if DO_EMULATION else load_task_manager()
validator = ReadoutOptimizerValidation(
    optimizer_run_dir=OPTIMIZER_RUN_DIR,
    profile=validation_profile,
    task_manager=validation_task_manager,
    settings=ReadoutOptimizerValidationSettings(
        profile_name=PROFILE_NAME,
        active_reset_num=ACTIVE_RESET_NUM,
        task_status_poll_interval=TASK_STATUS_POLL_INTERVAL,
        do_emulation=DO_EMULATION,
        show_handler_output=SHOW_HANDLER_OUTPUT,
        states=STATES,
    ),
)
validation_result = validator.run()
print(f"Saved validation artifacts to {validation_result['validation_dir']}")


## Inspect Results

In [ ]:
validation_result["rows"][: min(10, len(validation_result["rows"]))]
